In [1]:
import hashlib
import os
import platform
import uuid

# 修改为自己的学号
STUDENT_ID = 20244080223

CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV")
print(f"[Conda 环境] {CONDA_ENV}")

MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]

RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号] {STUDENT_ID}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID] {RUN_ID}")

[Conda 环境] None
[学号] 20244080223
[机器指纹] 9262db88e33a32bc
[RUN_ID] 20244080223-9262db88e33a32bc


Python 正则表达式练习

In [2]:
import re

# 1. 检查字符串是否仅包含英文字母 a-z、A-Z 和数字 0-9
def is_alnum_only(s):
    return bool(re.fullmatch(r'[a-zA-Z0-9]+', s))

# 2. 匹配 a 后面跟着零个或多个 b
def match_a_bb(s):
    return bool(re.fullmatch(r'ab*', s))

# 3. 匹配 a 后面跟着一个或多个 b
def match_a_bplus(s):
    return bool(re.fullmatch(r'ab+', s))

# 4. 匹配以 a 开头、b 结尾，中间任意字符
def match_a_any_b(s):
    return bool(re.fullmatch(r'a.*b', s))

# 5. 删除 IP 各段前导零（全零段保留一个 0）
def strip_ip_leading_zeros(ip):
    return '.'.join(str(int(seg)) for seg in ip.split('.'))

# 6. 空格替换为下划线，下划线替换为空格（互换）
def swap_space_underscore(s):
    tmp = re.sub(r' ', '\x00', s)
    tmp = re.sub(r'_', ' ', tmp)
    return re.sub(r'\x00', '_', tmp)

# ---------- 测试 ----------
print("1:", is_alnum_only("abc123"), is_alnum_only("abc 123"))
print("2:", match_a_bb("a"), match_a_bb("abbb"), match_a_bb("abbc"))
print("3:", match_a_bplus("a"), match_a_bplus("abbb"), match_a_bplus("ac"))
print("4:", match_a_any_b("axxxb"), match_a_any_b("ab"), match_a_any_b("a"))
print("5:", strip_ip_leading_zeros("192.168.001.010"),
      strip_ip_leading_zeros("010.000.000.001"))
print("6:", swap_space_underscore("a b_c d"))

1: True False
2: True True False
3: False True False
4: True True False
5: 192.168.1.10 10.0.0.1
6: a_b c_d


In [3]:
# 7. yyyy-mm-dd -> dd-mm-yyyy
def convert_date(s):
    m = re.fullmatch(r'(\d{4})-(\d{2})-(\d{2})', s)
    return f"{m.group(3)}-{m.group(2)}-{m.group(1)}" if m else s

# 8. 所有以 a 或 e 开头的单词
def words_start_ae(s):
    return re.findall(r'\b[aeAE]\w*', s)

# 9. Road 缩写为 Rd
def abbreviate_road(s):
    return re.sub(r'\bRoad\b', 'Rd', s)

# 10. 长度至少 4 个字符的单词
def words_len_ge4(s):
    return re.findall(r'\b\w{4,}\b', s)

# 11. 删除除字母和数字之外的所有字符
def remove_non_alnum(s):
    return re.sub(r'[^a-zA-Z0-9]', '', s)

# 12. 删除括号及括号内内容
def remove_parentheses(s):
    return re.sub(r'\s*\([^)]*\)', '', s).strip()

# ---------- 测试 ----------
print("7:", convert_date("2024-03-15"))
print("8:", words_start_ae("an apple and an egg"))
print("9:", abbreviate_road("Main Road, Oak Road"))
print("10:", words_len_ge4("I love Python and code"))
print("11:", remove_non_alnum("a1! b2@ c3#"))
for line in ["example (.com)", "w3resource", "github (.com)", "stackoverflow (.com)"]:
    print("12:", remove_parentheses(line))

7: 15-03-2024
8: ['an', 'apple', 'and', 'an', 'egg']
9: Main Rd, Oak Rd
10: ['love', 'Python', 'code']
11: a1b2c3
12: example
12: w3resource
12: github
12: stackoverflow


从网页检索气候数据

In [4]:
import requests
from bs4 import BeautifulSoup

def _get_soup(url):
    headers = {"User-Agent": "Mozilla/5.0 (edu homework)"}
    resp = requests.get(url, headers=headers, timeout=30)
    resp.raise_for_status()          # 网络/HTTP 错误抛出异常
    return BeautifulSoup(resp.text, "html.parser")

def _norm(text):
    return re.sub(r'\s+', ' ', text).strip()

In [5]:
def retrieve_climate_table(s):
    soup = _get_soup(s)
    # 找所有表格
    for table in soup.find_all("table"):
        # 国内天气网站通常没有 "Climate data for" 的 caption
        # 但气候/天气数据表格通常包含大量的行(tr)和列(th/td)
        rows = table.find_all("tr")
        if len(rows) > 10: # 随便设个阈值，大于10行的表格大概率是气候数据
            return table
    return None

In [6]:
def list_climate_table_row_names(s):
    """返回行名列表；成功读取但无表格返回 None。"""
    table = retrieve_climate_table(s)
    if table is None:
        return None
    names = []
    for tr in table.find_all("tr"):
        ths = tr.find_all("th")
        if not ths:
            continue
        row_th = None
        for th in ths:
            if th.get("scope") == "row":
                row_th = th
                break
        if row_th is None:
            first = ths[0]
            txt = _norm(first.get_text())
            if txt.startswith("Climate data for"):
                continue
            if len(ths) >= 12 and all(len(_norm(t.get_text())) <= 4 for t in ths):
                continue
            row_th = first
        txt = _norm(row_th.get_text())
        if not txt or txt.startswith("Climate data for"):
            continue
        if txt.lower().startswith(("source", "average", "mean")):
            continue
        names.append(txt)
    return names

In [7]:
test_urls = [
    "http://www.tianqihoubao.com/lishi/beijing/month/202310.html",
    "http://www.tianqihoubao.com/lishi/shanghai/month/202310.html"
]
for url in test_urls:
    try:
        t = retrieve_climate_table(url)
        print(url, "->", "found" if t else "None")
        if t:
            children = t.find_all(recursive=False)
            print("  直接子节点数:", len(children),
                  "标签:", [c.name for c in children][:10])
            names = list_climate_table_row_names(url)
            print("  行名:", names[:8], "...")
    except Exception as e:
        print(url, "-> ERROR:", type(e).__name__, e)

http://www.tianqihoubao.com/lishi/beijing/month/202310.html -> found
  直接子节点数: 2 标签: ['thead', 'tbody']
  行名: ['日期'] ...
http://www.tianqihoubao.com/lishi/shanghai/month/202310.html -> ERROR: ReadTimeout HTTPSConnectionPool(host='www.tianqihoubao.com', port=443): Read timed out. (read timeout=30)
